# Thực nghiệm bài toán tham quan Hà Nội trong một ngày

> Thêm khối trích dẫn



Notebook này chạy các thuật toán Greedy, A* và vét cạn được viết trong `greedy.py`, `astar.py`, `brute.py`; các hàm thực nghiệm nằm trong `experiments.py`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from experiments import run_all, OUTPUT_DIR, TIME_REPEATS, RANDOM_DATASETS, RANDOM_SEED
print("Thư mục dự án:", PROJECT_DIR)
print("Số lần đo thời gian cho TN1-TN2:", TIME_REPEATS)
print("Số bộ dữ liệu ngẫu nhiên TN3:", RANDOM_DATASETS)
print("Seed:", RANDOM_SEED)

## 1. Chạy toàn bộ thí nghiệm

Cell này sẽ chạy TN1, TN2 và TN3. TN1–TN2 đo thời gian trung bình qua 100 lần chạy mỗi thuật toán; TN3 chạy một lần trên từng bộ dữ liệu để hoàn thành 300 bộ dữ liệu ngẫu nhiên trong thời gian hợp lý. Các file CSV được lưu trong thư mục `results`.

In [ ]:
fixed_rows, random_rows, random_summary = run_all(
    repeats=TIME_REPEATS,
    n_random=RANDOM_DATASETS,
    seed=RANDOM_SEED
)

fixed_df = pd.DataFrame(fixed_rows)
random_df = pd.DataFrame(random_rows)
summary_df = pd.DataFrame([random_summary])

print("Đã chạy xong các thí nghiệm.")
print("Thư mục CSV:", OUTPUT_DIR)

## 2. Kết quả TN1 và TN2

`num_visited` là số điểm tham quan được ghé; `finish_clock` là giờ hoàn thành. `states` được định nghĩa theo thuật toán: Greedy là tổng số lần gọi `try_visit` (lọc ứng viên và đếm h), A* là số trạng thái được mở rộng, vét cạn là số hoán vị được đánh giá.

In [ ]:
display(fixed_df[[
    "scenario", "algorithm", "num_visited", "finish_clock",
    "states", "runtime_seconds_avg", "runtime_repeats", "path_names"
]])

In [ ]:
# Kiểm tra điều kiện nghiệm thu trên số điểm (mỗi dòng: TN1, TN2).
count_pivot = fixed_df.pivot(index="scenario", columns="algorithm", values="num_visited")
count_pivot["A*_equals_brute"] = count_pivot["A*"] == count_pivot["Vet can"]
count_pivot["brute_not_worse_than_greedy"] = count_pivot["Vet can"] >= count_pivot["Greedy"]
display(count_pivot)

## 3. Kết quả TN3 — 300 bộ dữ liệu ngẫu nhiên

Mỗi bộ dữ liệu thay đổi ma trận thời gian di chuyển, giờ mở cửa, giờ đóng cửa và thời gian tham quan theo mô tả trong file IDEA. Seed cố định giúp chạy lại có thể tái tạo cùng một bộ dữ liệu.

In [ ]:
display(summary_df)
display(random_df.head(10))

In [ ]:
# Biểu đồ phân bố độ chênh: số điểm vét cạn trừ số điểm Greedy.
gap_counts = random_df["gap_brute_minus_greedy"].value_counts().sort_index()
ax = gap_counts.plot(kind="bar", figsize=(8, 4))
ax.set_title("TN3: Phân bố độ chênh (Vét cạn - Greedy)")
ax.set_xlabel("Độ chênh số điểm")
ax.set_ylabel("Số bộ dữ liệu")
ax.set_xticklabels([str(x) for x in gap_counts.index], rotation=0)
plt.tight_layout()
plt.show()

## 4. Kiểm tra điều kiện nghiệm thu

Không chỉnh số liệu để ép kết quả đạt kỳ vọng. Nếu điều kiện nào không đạt, cần kiểm tra dữ liệu và thuật toán rồi chạy lại.

In [ ]:
tn2 = fixed_df[fixed_df["scenario"] == "TN2"].set_index("algorithm")
tn2_greedy = int(tn2.loc["Greedy", "num_visited"])
tn2_astar = int(tn2.loc["A*", "num_visited"])
tn2_brute = int(tn2.loc["Vet can", "num_visited"])

checks = {
    "A* bằng vét cạn ở mọi kịch bản TN1-TN2": bool((count_pivot["A*"] == count_pivot["Vet can"]).all()),
    "Vét cạn không kém Greedy ở TN1-TN2": bool((count_pivot["Vet can"] >= count_pivot["Greedy"]).all()),
    "TN2: Greedy ít hơn A* ít nhất 1 điểm": tn2_greedy < tn2_astar,
    "TN3: A* khác vét cạn bằng 0 bộ": random_summary["astar_differs_from_brute"] == 0,
}
for label, passed in checks.items():
    print(("ĐẠT" if passed else "CHƯA ĐẠT") + " — " + label)

if not checks["TN2: Greedy ít hơn A* ít nhất 1 điểm"]:
    print("\nLưu ý: với bộ dữ liệu hiện tại trong file IDEA, TN2 có thể cho Greedy bằng A* về số điểm. Không tự sửa kết quả; nhóm cần kiểm tra lại ma trận/giờ mở đóng cửa từ dữ liệu thực tế để tạo đúng kịch bản bẫy.")

In [ ]:
print("Các file kết quả:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", file.name)